In [ ]:
# Panggil fail solution.py yang kita buat tadi
from solution import STATES, get_group_states

# Tukar nombor dalam kurungan petak ini kepada nombor matrik sebenar ahli kumpulan anda!
nombor_matrik_kumpulan = [311279, 311294, 311397, 312114, 313429]

# Jalankan arahan untuk cari negeri
GROUP_STATES = get_group_states(nombor_matrik_kumpulan, STATES)

# Paparkan hasil negeri fokus
print("Negeri fokus kami ialah:", GROUP_STATES)

# Task 1
import json
from solution import validate_records, SURVEY_YEARS

# 1. Buka fail JSON dan ambil senarai di dalam kunci "records"
with open("data/raw/hh_income_state.json", "r") as f:
    data_inc = json.load(f)
    income_records = data_inc["records"] if isinstance(data_inc, dict) and "records" in data_inc else data_inc

with open("data/raw/hh_poverty_state.json", "r") as f:
    data_pov = json.load(f)
    poverty_records = data_pov["records"] if isinstance(data_pov, dict) and "records" in data_pov else data_pov

# 2. Tentukan kunci wajib
income_keys = ("state", "date", "income_median", "income_mean")
poverty_keys = ("state", "date", "poverty_absolute")

# 3. Jalankan fungsi semakan validate_records
validate_records(income_records, income_keys)
validate_records(poverty_records, poverty_keys)
print("Validation berjaya! Tiada ralat atau rekod bertindih dijumpai.")

# 4. Laporan Bilangan Rekod & Julat Tahun
years_income = sorted(list(set(int(r["date"][:4]) for r in income_records)))
print(f"\n--- Laporan Data Pendapatan ---")
print(f"• Jumlah rekod: {len(income_records)}")
print(f"• Tahun pertama: {years_income[0]} | Tahun terakhir: {years_income[-1]}")
print(f"• Bilangan tahun unik: {len(years_income)}")

# 5. Kira bilangan nilai None
none_counts_income = {key: sum(1 for r in income_records if r.get(key) is None) for key in income_keys}
print(f"• Bilangan nilai None dalam pendapatan: {none_counts_income}")

# 6. Tapis rekod untuk SURVEY_YEARS [2016, 2019, 2020, 2022, 2024]
filtered_income = [r for r in income_records if int(r["date"][:4]) in SURVEY_YEARS]

# 7. Semak kesemua 16 negeri wujud bagi setiap tahun tinjauan
for year in SURVEY_YEARS:
    states_in_year = set(r["state"] for r in filtered_income if int(r["date"][:4]) == year)
    if len(states_in_year) != 16:
        raise ValueError(f"Tahun {year} tidak mempunyai 16 negeri yang lengkap!")

print("Semakan lengkap: Kesemua 16 negeri wujud bagi setiap tahun dalam SURVEY_YEARS.")

Negeri fokus kami ialah: ['W.P. Putrajaya', 'W.P. Labuan', 'Pahang', 'Kelantan', 'Perak']
Validation berjaya! Tiada ralat atau rekod bertindih dijumpai.

--- Laporan Data Pendapatan ---
• Jumlah rekod: 319
• Tahun pertama: 1970 | Tahun terakhir: 2024
• Bilangan tahun unik: 22
• Bilangan nilai None dalam pendapatan: {'state': 0, 'date': 0, 'income_median': 11, 'income_mean': 0}
Semakan lengkap: Kesemua 16 negeri wujud bagi setiap tahun dalam SURVEY_YEARS.


In [ ]:
# Task 2: Bina Matriks NumPy
import numpy as np
from solution import STATES, SURVEY_YEARS, POVERTY_YEARS, index_by_key, join_records, to_float

joined_recs, income_only, poverty_only = join_records(income_records, poverty_records)
print(f"Jumlah rekod berjaya dipadankan: {len(joined_recs)}")
print(f"Kunci hanya dalam pendapatan: {len(income_only)}")
print(f"Kunci hanya dalam kemiskinan: {len(poverty_only)}")

income_indexed = index_by_key(income_records)
poverty_indexed = index_by_key(poverty_records)

median_list = []
for state in STATES:
    row = []
    for year in SURVEY_YEARS:
        rec = income_indexed.get((state, year))
        val = rec["income_median"] if rec else None
        row.append(to_float(val))
    median_list.append(row)
MEDIAN = np.array(median_list)

mean_list = []
for state in STATES:
    row = []
    for year in SURVEY_YEARS:
        rec = income_indexed.get((state, year))
        val = rec["income_mean"] if rec else None
        row.append(to_float(val))
    mean_list.append(row)
MEAN = np.array(mean_list)

poverty_list = []
for state in STATES:
    row = []
    for year in POVERTY_YEARS:
        rec = poverty_indexed.get((state, year))
        val = rec.get("poverty_absolute") if rec else None
        row.append(to_float(val))
    poverty_list.append(row)
POVERTY = np.array(poverty_list)

print("\n--- Bentuk Matriks Task 2 ---")
print("MEDIAN shape:", MEDIAN.shape)
print("MEAN shape:", MEAN.shape)
print("POVERTY shape:", POVERTY.shape)

Jumlah rekod berjaya dipadankan: 307
Kunci hanya dalam pendapatan: 12
Kunci hanya dalam kemiskinan: 3

--- Bentuk Matriks Task 2 ---
MEDIAN shape: (16, 5)
MEAN shape: (16, 5)
POVERTY shape: (16, 4)


Task 4.1 – Mean and Median of State Medians

In [ ]:
mean_state_medians = np.mean(MEDIAN, axis=0)
median_state_medians = np.median(MEDIAN, axis=0)

print("=== Task 4.1 ===")

for year, mean_val, median_val in zip(
    SURVEY_YEARS,
    mean_state_medians,
    median_state_medians
):
    print(
        f"{year}: Mean={mean_val:.2f}, Median={median_val:.2f}"
    )

=== Task 4.1 ===
2016: Mean=5235.94, Median=4636.50
2019: Mean=5915.12, Median=5275.00
2020: Mean=5276.19, Median=4634.00
2022: Mean=6212.69, Median=5552.00
2024: Mean=6741.94, Median=6109.00


These values are calculated from the 16 state medians only. They are not Malaysia's actual median household income because states have different population sizes.

Task 4.2 Percentage Growth in Median Income (2016–2024)

In [ ]:
growth = (
    (MEDIAN[:, -1] - MEDIAN[:, 0])
    / MEDIAN[:, 0]
) * 100

print("=== Task 4.2 ===")

for state, value in zip(STATES, growth):
    print(f"{state}: {value:.2f}%")

=== Task 4.2 ===
Johor: 36.45%
Kedah: 28.44%
Kelantan: 32.61%
Melaka: 23.32%
Negeri Sembilan: 22.10%
Pahang: 25.03%
Perak: 17.00%
Perlis: 17.75%
Pulau Pinang: 36.55%
Sabah: 18.98%
Sarawak: 32.21%
Selangor: 48.46%
Terengganu: 41.18%
W.P. Kuala Lumpur: 19.06%
W.P. Labuan: 24.54%
W.P. Putrajaya: 30.14%


In [ ]:
highest_idx = np.argmax(growth)
lowest_idx = np.argmin(growth)

print(
    f"Highest Growth State: {STATES[highest_idx]} ({growth[highest_idx]:.2f}%)"
)

print(
    f"Lowest Growth State: {STATES[lowest_idx]} ({growth[lowest_idx]:.2f}%)"
)


Highest Growth State: Selangor (48.46%)
Lowest Growth State: Perak (17.00%)


Task 4.3 Highest and Lowest Median Income State for Each Year

In [ ]:
highest_states = np.argmax(
    MEDIAN,
    axis=0
)

lowest_states = np.argmin(
    MEDIAN,
    axis=0
)

print("=== Task 4.3 ===")

for i, year in enumerate(SURVEY_YEARS):

    print(
        f"{year}: "
        f"Highest = {STATES[highest_states[i]]}, "
        f"Lowest = {STATES[lowest_states[i]]}"
    )

=== Task 4.3 ===
2016: Highest = W.P. Kuala Lumpur, Lowest = Kelantan
2019: Highest = W.P. Kuala Lumpur, Lowest = Kelantan
2020: Highest = W.P. Putrajaya, Lowest = Kelantan
2022: Highest = W.P. Kuala Lumpur, Lowest = Kelantan
2024: Highest = W.P. Kuala Lumpur, Lowest = Kelantan


Task 4.4 Relative Income Index

In [ ]:
state_median_reference = np.median(
    MEDIAN,
    axis=0
)

RELATIVE_INDEX = (
    MEDIAN
    / state_median_reference
)

print("=== Task 4.4 ===")
print(RELATIVE_INDEX)

=== Task 4.4 ===
[[1.21902297 1.21838863 1.22788088 1.23901297 1.26239974]
 [0.82195622 0.81990521 0.82628399 0.79286744 0.8012768 ]
 [0.66407851 0.67545024 0.64954683 0.6509366  0.66835816]
 [1.20521945 1.14767773 1.17544238 1.11851585 1.12800786]
 [0.9875984  0.94881517 0.96633578 0.94128242 0.91520707]
 [0.85819045 0.84170616 0.87030643 0.8560879  0.81437224]
 [0.8640138  0.81004739 0.81117825 0.80943804 0.76722868]
 [0.90671843 0.87090047 0.87246439 0.84888329 0.81027991]
 [1.16661275 1.16947867 1.18126888 1.17110951 1.20903585]
 [0.88644452 0.8028436  0.8141994  0.82438761 0.80045834]
 [0.89787555 0.8614218  0.82671558 0.89661383 0.90096579]
 [1.5582875  1.5563981  1.5753129  1.79809078 1.75577018]
 [1.0124016  1.05118483 1.03366422 1.05871758 1.08479293]
 [1.95686401 1.99981043 1.96223565 1.84329971 1.76821084]
 [1.27855063 1.27507109 1.32196806 1.24351585 1.20854477]
 [1.78475143 1.89251185 2.10250324 1.81123919 1.76280897]]


Task 4.5 States Below 0.8 and States with Lower Index in 2024

In [ ]:
below_08_mask = (
    RELATIVE_INDEX[:, -1] < 0.8
)

below_08_states = np.array(STATES)[below_08_mask]

print("States below 0.8 in 2024:")

for state in below_08_states:
    print(state)

States below 0.8 in 2024:
Kelantan
Perak


In [ ]:
lower_than_2016_mask = (
    RELATIVE_INDEX[:, -1]
    <
    RELATIVE_INDEX[:, 0]
)

lower_than_2016_states = np.array(STATES)[
    lower_than_2016_mask
]

print(
    "\nStates whose index is lower in 2024 than 2016:"
)

for state in lower_than_2016_states:
    print(state)



States whose index is lower in 2024 than 2016:
Kedah
Melaka
Negeri Sembilan
Pahang
Perak
Perlis
Sabah
W.P. Kuala Lumpur
W.P. Labuan
W.P. Putrajaya
